<a href="https://colab.research.google.com/github/Dania-Yasir/flyrak-project/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

## Search Impressions Decline — final model summary

This notebook is the concise Week-5 model summary. The full executable training workflow is in [`w05_model_training_.ipynb`](./w05_model_training_.ipynb). The later [`w06_validation_audit.ipynb`](./w06_validation_audit.ipynb) stress-tests the same modeling setup and provides the primary robustness estimate used in the capstone claim.

**Prediction target:** `is_declining_proxy = 1` when average daily impressions during March 16–31 are more than 20% lower than average daily impressions during March 1–15.


## 1. Method choice and why

I compared Logistic Regression, Decision Tree, and Random Forest. This is a binary classification task, but the practical decision is a **ranking** problem: which pages should be reviewed first for decline risk.

The primary metric is **Precision@100**, with Precision@10, Average Precision, and ROC-AUC as supporting metrics. Logistic Regression is the readable reference, Decision Tree captures simple non-linear rules, and Random Forest can capture interactions between level and momentum signals.

The Week 4 hand-written rule remains the main non-ML baseline. A stronger **momentum-only baseline** is reported in the validation audit as an additional stress test; it was not used as a model feature or target.


## 2. Split design

Two protections are used together:

- **Time separation:** model features come from March 1–15; the outcome is measured on March 16–31.
- **Client separation:** clients are kept together, so a client cannot appear in both training and validation/test partitions.

Hyperparameters are selected with **GroupKFold on training clients only**. The final client holdout is not used to choose the model. This matters because pages from the same client share context, and row-level random validation can overstate generalization.


## 3. Week-5 training result

The original Week-5 training notebook selected **Random Forest before opening the final test set** because it had the strongest grouped cross-validation Precision@100.

| Model | Grouped CV P@100 | CV spread (std) |
|---|---:|---:|
| Random Forest | **66.4%** | 9.3% |
| Logistic Regression | 63.8% | 14.4% |
| Decision Tree | 61.2% | 7.7% |

On the original frozen 7-client holdout, the selected Random Forest measured **78% Precision@100**, compared with **46%** for the Week 4 rule on the same clients. That is a **32 percentage-point observed lift** on that one holdout.

I do **not** use 78% as the final capstone performance estimate because seven clients produce a single, sample-sensitive point estimate. The validation audit below is the stronger evidence.


## 4. Final interpretation after the robustness audit

The validation audit tested the same 12-feature setup across multiple unseen client groups. The primary estimate is now the **nested grouped validation** result, not the single holdout.

| Evaluation | Random Forest P@100 | Momentum-only P@100 | Week 4 rule P@100 |
|---|---:|---:|---:|
| 5-fold nested client validation | **73.6% ± 8.4%** | 71.6% ± 12.6% | 34.4% ± 7.0% |
| 10 repeated 7-client holdouts | **72.6% ± 10.5%** | 70.9% ± 10.5% | 36.3% ± 10.0% |

The nested mean base rate was **32.1%**. Random Forest ranged from **63% to 84%** across the five outer folds. The repeated-holdout RF range was **54% to 84%**.

The main conclusion is therefore narrower than the original 78% point estimate:

- ML **clearly improves on the original Week 4 hand-written rule**.
- Recent impression momentum is a very strong single signal and explains much of the ranking performance.
- Random Forest adds a **modest incremental top-100 gain** over momentum alone, while also improving the broader ranking metrics in the nested evaluation: Average Precision **0.491 vs 0.448** and ROC-AUC **0.672 vs 0.619**.
- Nested macro-client Precision@100 was **55.1%**, so performance is not uniform across clients.

Random Forest remains the final ML model because it had the strongest mean Precision@100 in nested client-level validation and better broader ranking metrics than the momentum-only benchmark.


In [ ]:
import numpy as np
import pandas as pd

# Fold scores from the completed robustness audit.
nested = pd.DataFrame({
    "Random Forest": [0.84, 0.79, 0.63, 0.68, 0.74],
    "Momentum-only Baseline": [0.87, 0.83, 0.66, 0.59, 0.63],
    "Week 4 Baseline": [0.33, 0.45, 0.37, 0.27, 0.30],
})

summary = pd.DataFrame({
    "mean_p100": nested.mean(),
    "std_p100": nested.std(ddof=1),
    "min_p100": nested.min(),
    "max_p100": nested.max(),
})
display(summary.style.format("{:.1%}"))


## 5. Error analysis and interpretation

The strongest model feature was **`imp_momentum_log`**, followed by **`log_imp_first_half`** and **`imp_momentum_pct`** in the robustness run. The momentum ablation also showed that adding momentum to level features improved grouped Logistic Regression Precision@100 from **48.6% to 60.4%**.

Failure examples were consistent with that dependence. Some high-risk false positives had severe pre-outcome momentum drops but later recovered or stabilized. Some missed declines had weak or positive pre-outcome momentum, so there was little warning signal at prediction time.

This is why the model should be used as a **decision-support ranking tool**. A strong warning signal does not guarantee decline, and some declines occur without a strong warning signal.


## 6. Final claim and self-check

**Final claim:** In this experiment, the selected Random Forest ranked the defined future impression-decline proxy substantially better than the original Week 4 hand-written rule on unseen clients. Across five nested client-grouped evaluations, it measured **73.6% ± 8.4% Precision@100**. A momentum-only benchmark measured **71.6% ± 12.6%**, showing that momentum explains much of the signal and that the ML model adds a modest incremental gain rather than replacing a weak baseline with magic.

The result is **observed and directional**. It does not prove that a page will decline, establish causality, or model Google's ranking algorithm.

### Self-check

- [x] Features are from March 1–15 only.
- [x] Outcome is from March 16–31.
- [x] Client/content IDs are not model features.
- [x] Hyperparameters are selected with client-grouped validation.
- [x] The Week 4 rule and momentum-only benchmark are reported honestly.
- [x] The single 7-client result is not presented as the stable final average.
- [x] Final claims use observed / measured / directional / decision-support language.
